# 01 — Methodology: transformer internals from scratch

This notebook walks through the machinery FactLens is built on, with no
interpretability library in sight: **plain `nn.Module` forward hooks** over a
HuggingFace checkpoint. We will:

1. resolve GPT-2's blocks / final norm / unembedding in an architecture-agnostic way,
2. locate the **subject span** of a factual prompt via tokenizer offset mapping,
3. capture the **residual stream** at every site in one forward pass,
4. apply the **logit lens** by hand (final norm + unembedding), and
5. run one **activation patch** (clean -> corrupt -> patched) and watch the logit
   difference restore.

Prerequisites: `pip install -e .` in the repo root; the first cell downloads
GPT-2 (124M, ~500 MB).

In [ ]:
import torch

from factlens.models.loader import load_model_and_tokenizer
from factlens.models.hooks import ModuleMap

model, tokenizer = load_model_and_tokenizer('gpt2', dtype='float32', device='cpu')
mm = ModuleMap(model)

print(f'layers={mm.num_layers}  d_model={mm.d_model}  vocab={mm.vocab_size}')
print(f'final norm: {type(mm.final_norm).__name__} (kind={mm.norm_kind}, eps={mm.norm_eps})')
print(f'first/last block: {mm.block_names[0]} / {mm.block_names[-1]}')
print(f'residual sites: {mm.residual_sites()}')

## The fact bank

Each relation provides several **surface templates** per fact. The canonical
template (index 0) is what the lens and patching analyses use; all templates
feed the probes, with the *last* one held out as the probe test split — so probe
scores measure whether the fact is decodable across surface forms, not whether
the probe memorized a wording.

In [ ]:
from factlens.data.relations import get_fact_bank
from factlens.data.dataset import FactDataset

relations = get_fact_bank()  # all 9 relations; pass names to subset
dataset = FactDataset(relations=relations, tokenizer=tokenizer, prepend_bos=False)
print(dataset.describe())

In [ ]:
ex = dataset.example('capital-country', 'France', template_idx=0)
print('prompt           :', repr(ex.prompt))
print('input_ids        :', ex.input_ids)
print('subject span     :', ex.subject_token_span, '-> last subject token at', ex.subject_end_pos)
print('answer encoding  :', tokenizer.convert_ids_to_tokens(ex.answer_ids), '->', ex.answer_ids)

tokens = tokenizer.convert_ids_to_tokens(ex.input_ids)
start, end = ex.subject_token_span
marked = ['[', *tokens[start:end], ']']
print('tokenized        :', ' '.join(tokens[:start] + marked + tokens[end:]))

## Capturing the residual stream with hooks

FactLens indexes the residual stream at **sites** `embed, block1, ..., blockL` —
site `block k` is the *output* of the k-th block. One forward pass with a capture
hook per site yields the whole stack of hidden states:

In [ ]:
from factlens.models.hooks import make_capture_many, run_with_hooks

ids = torch.tensor([ex.input_ids])
store = {}
hooks = make_capture_many(mm, mm.residual_sites(), store)
out = run_with_hooks(model, ids, hooks)

for site in ['embed', 'block1', 'block8', 'block12']:
    print(f'{site:>8}: shape {tuple(store[site].shape)}, '
          f'norm at readout {store[site][0, ex.last_pos].norm():.1f}')
print(f'final logits shape: {tuple(out.logits.shape)}')

Note the norms: the residual stream **grows** with depth (the classic picture
from *Transformer Feed-Forward Layers Are Key-Value Memories*). Intuition
suggests rescaling each site's norm to the final site's before projecting —
but that is a **no-op**: LayerNorm and RMSNorm are invariant to positive
rescaling of their input, so the scalar cancels. The real lens distortion is
*translational* (intermediate residuals sit off-origin in directions the
unembedding never saw), which is what the tuned lens learns per layer.

FactLens's parameter-free stand-in is per-site **mean centering**:

$$\text{logits}^{(k)} = \text{Norm}\left(h^{(k)} - \mu^{(k)}\right) W_U^\top$$

with $\mu^{(k)}$ the dataset-level mean hidden vector at site $k$.

In [ ]:
from factlens.lens.logit_lens import LogitLens, estimate_site_means

# Mean-centered lens: subtract per-site dataset means before the final norm.
# (A scalar norm rescale would be a no-op: LayerNorm/RMSNorm are scale-invariant.)
analysis_examples = [
    dataset.example(rel.name, rel.facts[0][0], 0) for rel in relations
]
site_means = estimate_site_means(model, mm, analysis_examples, device='cpu')
lens = LogitLens(mm, site_means=site_means)

print(f"{'site':>8} | top-5 tokens under the lens")
print('-' * 66)
for site in mm.residual_sites() + ['final']:
    if site == 'final':
        logits = out.logits[0, ex.last_pos].float()
    else:
        logits = lens.project(store[site][0, ex.last_pos], site=site)
    top = logits.topk(5)
    words = [f'{tokenizer.decode([t]):>10}' for t in top.indices]
    print(f'{site:>8} | ' + ' '.join(words))


Watch ` Paris` climb: somewhere mid-stack the answer surfaces in the
unembedding space long before the model commits to it. Scanning every fact and
aggregating these ranks is `run_lens` (notebook 02).

## One activation patch, by hand

Corrupt the prompt by swapping the subject for a counterfactual
(`France -> Germany`), then splice the **clean** site-8 representation at the
last subject token back into the corrupt run. If that site carries the fact, the
answer logit difference should jump back toward the clean value.

In [ ]:
from factlens.models.hooks import make_patch_hook

counter = dataset.counterfactual(ex)   # same template, subject swapped
print('clean    :', repr(ex.prompt),      '-> answer', repr(' ' + ex.object))
print('corrupt  :', repr(counter.prompt), '-> answer', repr(' ' + counter.object))

ids_corr = torch.tensor([counter.input_ids])
ans, dis = ex.answer_token_id, counter.answer_token_id

def logit_diff(logits):
    row = logits[0, -1]
    return float(row[ans] - row[dis])

ld_clean = logit_diff(out.logits)
out_corr = run_with_hooks(model, ids_corr, [])
ld_corr = logit_diff(out_corr.logits)
print(f'\nld(clean) = {ld_clean:+.2f}   ld(corrupt) = {ld_corr:+.2f}')

site = 'block8'
hook = make_patch_hook(store[site], [ex.subject_end_pos], [counter.subject_end_pos])
out_patched = run_with_hooks(model, ids_corr, [(mm.resolve_site(site), hook)])
ld_patched = logit_diff(out_patched.logits)
restoration = (ld_patched - ld_corr) / (ld_clean - ld_corr)
print(f'patching {site} at the last subject token: ld = {ld_patched:+.2f} '
      f'-> restoration {restoration:.2f}')

## What you just built

Everything the pipelines do is these three primitives, repeated and aggregated:

| primitive | pipeline | question answered |
|---|---|---|
| lens projection | `run_lens` | *where does the answer surface in vocab space?* |
| linear probe | `run_probes` | *where is the fact linearly decodable?* |
| patch | `run_patching` | *where does the model actually use it?* |

Notebook 02 runs the lens at scale; notebook 03 combines probes and patches into
the **Causal-Probe Gap**.